# BloodMNIST validation-only staged Colab run

Execute top to bottom on a Google Colab NVIDIA T4. Each stage calls the repository's modular Python APIs; no data, model, training, or metric implementation is duplicated here. The locked test split is never requested.


## 1. Runtime and source provenance

Fail closed on a non-T4 runtime and on an unset/unreviewed source commit.


In [ ]:
import re
import torch
assert torch.cuda.is_available(), "Select a Google Colab NVIDIA T4 runtime."
gpu_name = torch.cuda.get_device_name(0)
assert "T4" in gpu_name, f"T4 required; got {gpu_name}"
print("GPU:", gpu_name)


In [ ]:
REPO_URL = "https://github.com/james-chula/deep_learning_term_project.git"
COMMIT = "6b71dff8ea41fb599cc33497de88c56700900881"
assert len(COMMIT) == 40 and re.fullmatch(r"[0-9a-f]{40}", COMMIT), "Set COMMIT to the Lead-approved full SHA."
!rm -rf /content/project
!git clone -q $REPO_URL /content/project
%cd /content/project
!git checkout --detach $COMMIT
!pip install -q -r requirements-colab.txt
actual_commit = !git rev-parse HEAD
assert actual_commit[0] == COMMIT
print("Pinned commit:", actual_commit[0])
# The unchanged command-line entry point remains: python run_experiment.py


## 2. Configuration

Edit only the approved plain dictionary. The notebook rejects unknown modes and records the exact dictionary in the run artifacts.


In [ ]:
import os
os.environ["MEDMNIST_DATA_ROOT"] = "/content/medmnist"
import sys
sys.path.insert(0, "/content/project/src")
sys.path.insert(0, "/content/project")

from config import CONFIG
CONFIG.update({
    "mode": "finetune",  # majority | linear_probe | finetune
    "run_name": "bloodmnist_finetune_1pct",
    "use_fixed_1pct_subset": True,
    "seed": 2026,
    "epochs": 10,
    "batch_size": 64,
    "learning_rate": 0.001,
})
assert CONFIG["mode"] in {"majority", "linear_probe", "finetune"}
CONFIG


## 3. Audited data and sample visualization

The canonical provider returns only training and validation datasets. This stage verifies its locked-test metadata, prints provenance/counts, and visualizes training samples.


In [ ]:
import json
import matplotlib.pyplot as plt
from data import get_data_loaders
from run_experiment import class_names_from_audit, display_class_name, set_seed

set_seed(CONFIG["seed"])
train_loader, validation_loader, provider_metadata = get_data_loaders(CONFIG)
assert provider_metadata["test_policy"] == "locked_not_loaded"
class_names = class_names_from_audit(CONFIG["audit_manifest_path"])
print(json.dumps(provider_metadata, indent=2))
print("loader counts:", len(train_loader.dataset), len(validation_loader.dataset))

images, labels = next(iter(train_loader))
mean = torch.tensor((0.485, 0.456, 0.406)).view(3, 1, 1)
std = torch.tensor((0.229, 0.224, 0.225)).view(3, 1, 1)
figure, axes = plt.subplots(2, 4, figsize=(12, 6))
for axis, image, label in zip(axes.flat, images[:8], labels.view(-1)[:8]):
    axis.imshow((image.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0))
    axis.set_title(display_class_name(label, class_names))
    axis.axis("off")
figure.suptitle("Audited training samples")
plt.tight_layout()
plt.show()


## 4. Model summary

Majority mode has no neural model. Neural modes use the shared ImageNet-pretrained ResNet-18 builder; the trainable-parameter count confirms linear-probe freezing versus full fine-tuning.


In [ ]:
from model import create_model

if CONFIG["mode"] == "majority":
    model_preview = None
    print("Majority baseline: no neural model or optimizer.")
else:
    model_preview = create_model(CONFIG["num_classes"], CONFIG["mode"])
    total_parameters = sum(parameter.numel() for parameter in model_preview.parameters())
    trainable_parameters = sum(parameter.numel() for parameter in model_preview.parameters() if parameter.requires_grad)
    print(model_preview)
    print({"total_parameters": total_parameters, "trainable_parameters": trainable_parameters})
del model_preview


## 5. Training setup and evidence directory

This stage creates the run folder and writes provenance before any optimization. Neural checkpoint selection is validation macro-F1 only.


In [ ]:
import time
from pathlib import Path
from run_experiment import (current_commit, make_output_folder, package_versions,
                            require_t4, save_json)

assert current_commit() == COMMIT
set_seed(CONFIG["seed"])  # reset after sample/model-preview cells for reproducible training
started = time.time()
output_folder = make_output_folder()
run_metadata = {
    "config": dict(CONFIG),
    "provider": provider_metadata,
    "git_commit": current_commit(),
    "gpu": require_t4(),
    "package_versions": package_versions(),
    "test_evaluated": False,
    "test_policy": "locked_not_loaded",
}
save_json(output_folder / "run_metadata.json", run_metadata)
save_json(output_folder / "config.json", CONFIG)
print("Evidence directory:", output_folder)
print("Selection metric: validation macro-F1")


## 6. Training

Majority evaluates a training-derived constant predictor once. Neural modes train and atomically save best/last checkpoints through the shared API.


In [ ]:
from model import majority_probabilities
from train import validate_majority
from run_experiment import train_neural_mode

device = torch.device("cuda")
if CONFIG["mode"] == "majority":
    train_labels = []
    for _, batch_labels in train_loader:
        train_labels.extend(batch_labels.view(-1).tolist())
    probabilities, majority_class = majority_probabilities(
        train_labels, len(validation_loader.dataset), CONFIG["num_classes"])
    selected_result = validate_majority(probabilities, validation_loader)
    selected_result["probabilities"] = probabilities.numpy()
    best_epoch = None
    history = []
    print("Majority class:", majority_class, class_names[majority_class])
else:
    best_epoch, best_f1, _, history = train_neural_mode(
        CONFIG["mode"], train_loader, validation_loader, device, output_folder)
    print({"selected_epoch": best_epoch, "validation_macro_f1": best_f1})


## 7. Learning curves

Neural modes display the already-saved selected-run curves. Majority mode correctly has no epoch curve.


In [ ]:
from IPython.display import Image, display
curve_path = output_folder / "learning_curves.png"
if curve_path.is_file():
    display(Image(filename=str(curve_path)))
else:
    print("No learning curve for majority mode.")


## 8. Reload selected checkpoint and validate

For neural modes this explicitly reloads `best_model.pt`, then recomputes validation outputs. Thus saved final metrics and diagnostics correspond to the selected checkpoint rather than the last epoch.


In [ ]:
from run_experiment import evaluate_best_checkpoint, validation_metrics

if CONFIG["mode"] != "majority":
    selected_model, checkpoint_epoch, selected_result = evaluate_best_checkpoint(
        CONFIG["mode"], validation_loader, device, output_folder)
    assert checkpoint_epoch == best_epoch
    del selected_model
metrics = validation_metrics(
    selected_result["labels"], selected_result["predictions"], selected_result.get("probabilities"))
metrics


## 9. Validation diagnostics

Generate count and normalized confusion matrices, per-class precision/recall/F1, and misclassified validation samples. These functions receive only the validation loader.


In [ ]:
from run_experiment import save_validation_diagnostics

save_validation_diagnostics(
    selected_result["labels"], selected_result["predictions"], metrics,
    validation_loader, class_names, output_folder)
for filename in ["confusion_matrix_counts.png", "confusion_matrix_normalized.png",
                 "per_class_metrics.png", "misclassified_samples.png"]:
    path = output_folder / filename
    if path.is_file():
        print(filename)
        display(Image(filename=str(path)))


## 10. Finalize and download artifact ZIP

Write final validation-only metrics/summary, then package the complete evidence directory.


In [ ]:
from run_experiment import create_artifact_zip

summary = {
    "mode": CONFIG["mode"],
    "best_epoch": best_epoch,
    "best_validation_macro_f1": metrics["macro_f1"],
    "runtime_seconds": time.time() - started,
    "test_evaluated": False,
}
if CONFIG["mode"] == "majority":
    summary["majority_class"] = majority_class
save_json(output_folder / "metrics.json", metrics)
save_json(output_folder / "summary.json", summary)
archive_path = create_artifact_zip(output_folder)
print(json.dumps(summary, indent=2))
print("Artifact ZIP:", archive_path)
from google.colab import files
files.download(str(archive_path))
